# IF25-40305: Sistem Teknologi Multimedia (STM)
## Modul 12 — Pose Estimation, Hand Tracking & Gesture Recognition: 33 Titik Tubuh, Sudut Sendi, 21 Titik Tangan, Hitung Jari & Virtual Painter

**Institut Teknologi Sumatera (ITERA) — Program Studi Informatika**
*Materi Kuliah & Hands-on Lab Semester Ganjil 2026/2027 · Pertemuan 12*

---

### 🎯 Capaian Pembelajaran Modul 12:
1. Mengambil **33 titik tubuh** (*pose landmarks*) dengan **MediaPipe Pose Landmarker**, dari foto maupun video.
2. Menghitung **sudut sendi** (siku, lutut) dari tiga titik, lalu memakainya untuk **menghitung repetisi squat** otomatis.
3. Mengambil **21 titik tangan** dengan **MediaPipe Hand Landmarker** dan memahami label tangan kiri/kanan.
4. Membuat aturan sederhana untuk **menghitung jari yang terangkat** dan membedakan **kepal vs telapak terbuka**.
5. Membandingkan aturan buatan sendiri dengan **Gesture Recognizer** bawaan MediaPipe.
6. Menggabungkan semuanya menjadi **Virtual Painter**: telunjuk menjadi kuas di udara.

### 🧭 Cara Memakai Notebook Ini
- Di kelas, dosen menjalankan bagian bertanda **🎬 Demo Dosen** di proyektor. Anda cukup memperhatikan dan menebak.
- Di rumah, jalankan notebook dari atas ke bawah. Sel bertanda **✏️ Coba Ubah** berisi angka yang boleh Anda ganti.
- Semua video, foto, dan model **diunduh otomatis** saat pertama kali dijalankan (±25 MB). Butuh internet sekali saja.
- Bagian 9 memakai dua script terpisah, **`12_pose_sudut.py`** dan **`12_virtual_painter.py`**, yang butuh webcam.
  Jalankan di laptop Anda (bukan di Colab).

> 📎 Slide pendamping: *Pertemuan 12 — Pose, Tangan & Gestur* (mctm.web.id/stm).

## 1. Persiapan

Modul ini memakai **`opencv-contrib-python`**, **`mediapipe`**, **`numpy`**, dan **`matplotlib`** — sama dengan Modul 11.

```bash
pip install opencv-contrib-python mediapipe numpy matplotlib pillow
```

**Bahan yang dipakai** (semuanya domain publik atau CC0, jadi bebas dipakai belajar):

| Berkas | Isi | Sumber & lisensi |
|---|---|---|
| `video/latihan_squat.webm` | Instruktur US Army memperagakan *Alternate Staggered Squat Jump* (±52 detik) | U.S. Army Physical Readiness Training, via Wikimedia Commons — domain publik |
| `video/hitung_jari.webm` | Seorang perempuan menghitung 1–10 dengan jari (±13 detik) | Mvolz, Wikimedia Commons — CC0 |
| `image/jari_dari_jempol.png` | 5 foto tangan menghitung 1–5 mulai dari **jempol** | Justinkunimune, Wikimedia Commons — CC0 |
| `image/jari_dari_telunjuk.png` | 5 foto tangan menghitung 1–5 mulai dari **telunjuk** | Justinkunimune, Wikimedia Commons — CC0 |
| `image/jari_menekuk.png` | 5 foto tangan **menekuk** jari satu per satu sampai mengepal | Justinkunimune, Wikimedia Commons — CC0 |
| `image/astronaut_melambai.jpg` | Astronaut Chris Cassidy melambai setelah mendarat (2013) | NASA/Bill Ingalls, via Wikimedia Commons — domain publik |
| `model/*.task` | Pose Landmarker, Hand Landmarker, Gesture Recognizer | Google MediaPipe — Apache-2.0 |

> 💡 Angka kecil di pojok foto jari (1, 2, 3, …) adalah **jawaban yang benar**. Nanti kita bandingkan dengan hitungan program.

In [ ]:
import os
import shutil
import warnings
import urllib.request
from collections import Counter, deque

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Image, display
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions

warnings.filterwarnings("ignore", category=RuntimeWarning)   # rata-rata dari frame kosong (NaN) -> diam saja
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Folder kerja: bisa dari root repo (ada folder notebook_handson) atau dari dalam notebook_handson / Colab
BASE = "notebook_handson" if os.path.isdir("notebook_handson") else "."
for d in ["image", "video", "model", "output"]:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)
print("mediapipe", mp.__version__, "| opencv", cv2.__version__, "| folder kerja:", os.path.abspath(BASE))

In [ ]:
# ---------- Unduh bahan (sekali saja; kalau sudah ada, dilewati) ----------
UA = {"User-Agent": "IF25-40305-handson/1.0 (course notebook; https://mctm.web.id/stm)"}
WM = "https://upload.wikimedia.org/wikipedia/commons/"
MP = "https://storage.googleapis.com/mediapipe-models/"

BAHAN = {
    "video/latihan_squat.webm": WM + "3/34/Conditioning_Drill_3-_Alternate_Staggered_Squat_Jump.webm",
    "video/hitung_jari.webm": WM + "e/e5/Woman_counting_on_fingers.webm",
    "image/jari_dari_jempol.png": WM + "5/56/Finger-counting_to_5_from_thumb.png",
    "image/jari_dari_telunjuk.png": WM + "a/a0/Finger-counting_to_5_from_pointer.png",
    "image/jari_menekuk.png": WM + "6/66/Finger-counting_to_5_tucking_from_thumb.png",
    "image/astronaut_melambai.jpg": WM + "9/9a/ISS-36_Cassidy_waves_to_support_personnel.jpg",
    "model/pose_landmarker_full.task": MP + "pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
    "model/hand_landmarker.task": MP + "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "model/gesture_recognizer.task": MP + "gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task",
}


def P(nama):
    return os.path.join(BASE, nama)


for nama, url in BAHAN.items():
    if not os.path.exists(P(nama)):
        print("mengunduh", nama, "...")
        req = urllib.request.Request(url, headers=UA)
        with urllib.request.urlopen(req) as r, open(P(nama), "wb") as f:
            shutil.copyfileobj(r, f)
    print(f"  ✓ {nama:34s} {os.path.getsize(P(nama)) / 1e6:5.1f} MB")

In [ ]:
# ---------- Fungsi bantu tampilan ----------
def tampil(ax, img, judul=None):
    if img.ndim == 2:
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    else:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if judul:
        ax.set_title(judul, fontsize=11, fontweight='bold')


def jajar(daftar, ukuran=4):
    # daftar = [(img, judul), ...] -> ditampilkan berdampingan
    fig, axes = plt.subplots(1, len(daftar), figsize=(ukuran * len(daftar), ukuran * 0.8))
    for ax, (im, judul) in zip(np.atleast_1d(axes), daftar):
        tampil(ax, im, judul)
    plt.tight_layout(); plt.show()


def tampil_gif(frames, nama, fps=24, lebar=360, langkah=2):
    # Simpan list frame (BGR) menjadi GIF animasi, lalu tampilkan di notebook
    tinggi = int(frames[0].shape[0] * lebar / frames[0].shape[1])
    gambar = [PILImage.fromarray(cv2.cvtColor(cv2.resize(f, (lebar, tinggi)), cv2.COLOR_BGR2RGB))
              for f in frames[::langkah]]
    path = P(os.path.join("output", nama))
    gambar[0].save(path, save_all=True, append_images=gambar[1:], duration=int(1000 * langkah / fps), loop=0)
    with open(path, "rb") as fh:
        display(Image(data=fh.read(), format="gif"))
    print(f"GIF disimpan: {path} ({os.path.getsize(path) / 1024:.0f} KB)")


def tulis(img, teks, xy, skala=0.7, warna=(255, 255, 255)):
    # teks dengan garis tepi hitam agar terbaca di latar apa pun
    cv2.putText(img, teks, xy, cv2.FONT_HERSHEY_SIMPLEX, skala, (0, 0, 0), 5, cv2.LINE_AA)
    cv2.putText(img, teks, xy, cv2.FONT_HERSHEY_SIMPLEX, skala, warna, 2, cv2.LINE_AA)


def baca_video(path, lebar=480, langkah=1):
    # Baca video menjadi list frame (BGR) selebar `lebar` piksel, ambil 1 dari tiap `langkah` frame
    # (supaya hemat memori: 1 menit video 30 fps = 1800 frame!)
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames, i = [], 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        if i % langkah == 0:
            s = lebar / f.shape[1]
            frames.append(cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA))
        i += 1
    cap.release()
    return frames, fps / langkah


def ke_mp(img):
    # OpenCV memakai BGR, MediaPipe butuh RGB dalam bungkus mp.Image
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


# Warna (BGR) — palet ITERA
EMAS, TEAL, MERAH, HIJAU, BIRU, ORANYE = (74, 195, 232), (140, 140, 30), (46, 16, 200), (107, 158, 46), (200, 110, 30), (57, 123, 224)

---
## 2. Tubuh Jadi 33 Titik (Pose Landmarker)

Minggu lalu wajah menjadi **478 titik**. Hari ini **seluruh tubuh** menjadi **33 titik** bernomor tetap:

| Nomor | Bagian | Nomor | Bagian |
|---|---|---|---|
| 0 | hidung | 23 / 24 | pinggul kiri / kanan |
| 11 / 12 | bahu kiri / kanan | 25 / 26 | lutut kiri / kanan |
| 13 / 14 | siku kiri / kanan | 27 / 28 | pergelangan kaki kiri / kanan |
| 15 / 16 | pergelangan tangan kiri / kanan | 29–32 | tumit & ujung kaki |

Pola mudah diingat: **nomor ganjil = kiri, nomor genap = kanan** — kiri dan kanan **pemilik tubuh**, bukan kiri-kanan layar.

Caranya mirip Face Landmarker: buat *landmarker*, beri gambar, terima daftar titik dengan `x, y` berupa **pecahan 0–1**.
Selain `x, y`, tiap titik punya `z` (kedalaman kira-kira) dan **`visibility`** (0–1: seberapa yakin titik itu terlihat).

In [ ]:
# Sambungan antar titik untuk menggambar "kerangka"
SAMBUNG_POSE = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16),            # bahu & lengan
                (11, 23), (12, 24), (23, 24),                                # badan
                (23, 25), (25, 27), (24, 26), (26, 28),                      # kaki
                (27, 29), (29, 31), (27, 31), (28, 30), (30, 32), (28, 32),  # telapak kaki
                (15, 17), (15, 19), (17, 19), (16, 18), (16, 20), (18, 20),  # telapak tangan
                (0, 2), (0, 5), (2, 7), (5, 8), (9, 10)]                     # wajah


def buat_pose(mode="IMAGE"):
    opsi = vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=P("model/pose_landmarker_full.task")),
        running_mode=getattr(vision.RunningMode, mode), num_poses=1)
    return vision.PoseLandmarker.create_from_options(opsi)


def titik_pose(hasil, w, h):
    # -> array (33, 3): x piksel, y piksel, visibility. None kalau tidak ada orang.
    if not hasil.pose_landmarks:
        return None
    return np.array([[p.x * w, p.y * h, p.visibility] for p in hasil.pose_landmarks[0]], np.float32)


def gambar_kerangka(img, titik, sambung, warna=EMAS, tebal=3, r=5, min_vis=0.5):
    out = img.copy()
    for a, b in sambung:
        if titik[a, 2] >= min_vis and titik[b, 2] >= min_vis:
            cv2.line(out, tuple(np.int32(titik[a, :2])), tuple(np.int32(titik[b, :2])), warna, tebal, cv2.LINE_AA)
    for x, y, v in titik:
        if v >= min_vis:
            cv2.circle(out, (int(x), int(y)), r, MERAH, -1, cv2.LINE_AA)
    return out


SQUAT, FPS_SQUAT = baca_video(P("video/latihan_squat.webm"), lebar=480, langkah=2)   # ±15 fps cukup untuk squat
H, W = SQUAT[0].shape[:2]
print(f"video squat: {len(SQUAT)} frame @ {FPS_SQUAT:.2f} fps = {len(SQUAT) / FPS_SQUAT:.1f} detik, ukuran {W}x{H}")

### 🎬 Demo Dosen — 33 titik pada satu frame
**Tebak dulu:** titik mana yang paling *tidak yakin* (visibility paling kecil)? Kenapa?

In [ ]:
# ✏️ Coba Ubah: nomor frame (0 sampai jumlah frame - 1)
NO_FRAME = len(SQUAT) // 3

pose_img = buat_pose("IMAGE")
f = SQUAT[NO_FRAME]
T = titik_pose(pose_img.detect(ke_mp(f)), W, H)
if T is None:
    print("Tidak ada orang di frame ini — coba nomor frame lain.")
else:
    bernomor = gambar_kerangka(f, T, SAMBUNG_POSE)
    for i in [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]:
        tulis(bernomor, str(i), (int(T[i, 0]) + 6, int(T[i, 1]) - 4), 0.45, EMAS)
    jajar([(f, f"Frame {NO_FRAME}"), (bernomor, "33 titik + kerangka")], 6)

    NAMA = {0: "hidung", 11: "bahu kiri", 12: "bahu kanan", 13: "siku kiri", 14: "siku kanan",
            15: "pergelangan kiri", 16: "pergelangan kanan", 23: "pinggul kiri", 24: "pinggul kanan",
            25: "lutut kiri", 26: "lutut kanan", 27: "pergelangan kaki kiri", 28: "pergelangan kaki kanan"}
    print(f"{'No':>3} {'bagian':24s} {'x':>6} {'y':>6} {'visibility':>10}")
    for i, nm in NAMA.items():
        print(f"{i:>3} {nm:24s} {T[i, 0]:6.0f} {T[i, 1]:6.0f} {T[i, 2]:10.2f}")

**Perhatikan:**
- Titik **kiri** (nomor ganjil) muncul di sisi **kanan layar** kalau orangnya menghadap kamera — seperti melihat orang lain, bukan cermin.
- Bagian yang tertutup tubuh sendiri biasanya punya `visibility` lebih kecil. Model tetap *menebak* posisinya.
- Satu frame `480×270` diproses dalam puluhan milidetik di CPU — cukup cepat untuk video *real-time*.

---
## 3. Sudut Sendi: Tiga Titik, Satu Engsel

Siku adalah **engsel**: lengan atas dan lengan bawah bertemu di sana. Kalau kita tahu tiga titik —
**bahu – siku – pergelangan** — kita bisa mengukur seberapa **terbuka** engselnya:

- lengan lurus ➜ sekitar **180°**
- lengan ditekuk siku-siku ➜ sekitar **90°**
- lengan dilipat penuh ➜ sekitar **30–40°**

Sama untuk lutut: **pinggul – lutut – pergelangan kaki**.

Tidak perlu menghafal rumus: fungsi di bawah membuat dua "tongkat" dari titik tengah, lalu mengukur sudut di antara keduanya
(persis seperti busur derajat di pelajaran SD).

In [ ]:
def sudut(a, b, c):
    # Sudut di titik b (derajat) yang dibentuk oleh tongkat b->a dan b->c
    ba = np.asarray(a[:2], float) - np.asarray(b[:2], float)
    bc = np.asarray(c[:2], float) - np.asarray(b[:2], float)
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return float(np.degrees(np.arccos(np.clip(cos, -1, 1))))


# Cek dengan titik buatan sendiri (sama dengan Aktivitas di slide)
print("lurus     :", round(sudut((0, 0), (10, 0), (20, 0))), "derajat")
print("siku-siku :", round(sudut((10, -10), (10, 0), (20, 0))), "derajat")
print("menekuk   :", round(sudut((20, -10), (10, 0), (20, 0))), "derajat")

In [ ]:
LUTUT_KIRI, LUTUT_KANAN = (23, 25, 27), (24, 26, 28)      # pinggul - lutut - pergelangan kaki
SIKU_KIRI, SIKU_KANAN = (11, 13, 15), (12, 14, 16)        # bahu - siku - pergelangan tangan


def gambar_sudut(img, titik, tiga, warna=TEAL):
    a, b, c = (titik[i, :2] for i in tiga)
    nilai = sudut(a, b, c)
    for p, q in [(a, b), (b, c)]:
        cv2.line(img, tuple(np.int32(p)), tuple(np.int32(q)), warna, 4, cv2.LINE_AA)
    cv2.circle(img, tuple(np.int32(b)), 8, EMAS, -1, cv2.LINE_AA)
    tulis(img, f"{nilai:.0f}", (int(b[0]) + 10, int(b[1]) + 5), 0.7, EMAS)
    return nilai


if T is not None:
    demo = f.copy()
    for tiga in [LUTUT_KIRI, LUTUT_KANAN, SIKU_KIRI, SIKU_KANAN]:
        gambar_sudut(demo, T, tiga)
    jajar([(demo, "Sudut lutut & siku (derajat)")], 7)

### 🎬 Demo Dosen — sudut lutut sepanjang video
**Tebak dulu:** berapa kali instruktur turun (*squat*) di video ini? Seperti apa bentuk grafik sudut lututnya?

Kita jalankan Pose Landmarker di **semua frame** dengan mode `VIDEO` (lebih cepat dan stabil karena memakai hasil frame sebelumnya).

In [ ]:
pose_vid = buat_pose("VIDEO")
SEMUA_T = []                                   # titik tiap frame (atau None)
for i, fr in enumerate(SQUAT):
    hasil = pose_vid.detect_for_video(ke_mp(fr), int(i * 1000 / FPS_SQUAT))
    SEMUA_T.append(titik_pose(hasil, W, H))
pose_vid.close()

waktu = np.arange(len(SQUAT)) / FPS_SQUAT


def deret_sudut(semua, tiga, min_vis=0.5):
    out = []
    for t in semua:
        if t is None or min(t[i, 2] for i in tiga) < min_vis:
            out.append(np.nan)                 # titik tidak terlihat -> kosong
        else:
            out.append(sudut(*(t[i] for i in tiga)))
    return np.array(out)


lutut = np.nanmean([deret_sudut(SEMUA_T, LUTUT_KIRI), deret_sudut(SEMUA_T, LUTUT_KANAN)], axis=0)
ada = np.mean([t is not None for t in SEMUA_T])
print(f"orang terdeteksi di {ada * 100:.0f}% frame")

fig, ax = plt.subplots(figsize=(12, 3.4))
ax.plot(waktu, lutut, color="#1E8C8C", lw=1.6)
ax.set_xlabel("waktu (detik)"); ax.set_ylabel("sudut lutut (°)")
ax.set_title("Rata-rata sudut lutut kiri & kanan", fontweight="bold")
plt.tight_layout(); plt.show()

### Menghitung repetisi = menghitung kedip versi tubuh
Ingat penghitung kedip minggu lalu? Polanya sama, hanya sekarang memakai **dua ambang**:

1. Mulai di posisi **berdiri** (lutut lurus).
2. Sudut lutut turun di bawah **ambang bawah** ➜ catat "sedang di bawah".
3. Sudut naik lagi di atas **ambang atas** ➜ **satu repetisi selesai**, kembali ke "berdiri".

Kenapa dua ambang, bukan satu? Kalau hanya satu (misalnya 130°), sudut yang **bergetar** di sekitar 130° akan
terhitung berkali-kali. Jarak antara dua ambang menjadi "zona aman" anti-getar.

In [ ]:
def hitung_repetisi(deret, bawah=110, atas=150):
    hitung, posisi, momen = 0, "atas", []
    for i, s in enumerate(deret):
        if np.isnan(s):
            continue
        if posisi == "atas" and s < bawah:
            posisi = "bawah"
        elif posisi == "bawah" and s > atas:
            posisi = "atas"
            hitung += 1
            momen.append(i)
    return hitung, momen


# ✏️ Coba Ubah: ambang bawah & atas. Lihat grafik: di mana lembah dan puncaknya?
AMBANG_BAWAH, AMBANG_ATAS = 110, 150
n, momen = hitung_repetisi(lutut, AMBANG_BAWAH, AMBANG_ATAS)
print(f"Repetisi terhitung: {n}")

fig, ax = plt.subplots(figsize=(12, 3.4))
ax.plot(waktu, lutut, color="#1E8C8C", lw=1.6)
ax.axhline(AMBANG_BAWAH, color="#C8102E", ls="--", lw=1, label=f"ambang bawah {AMBANG_BAWAH}°")
ax.axhline(AMBANG_ATAS, color="#D4A017", ls="--", lw=1, label=f"ambang atas {AMBANG_ATAS}°")
for k, i in enumerate(momen, 1):
    ax.annotate(str(k), (waktu[i], lutut[i]), xytext=(0, 8), textcoords="offset points",
                ha="center", fontsize=9, fontweight="bold", color="#0B1F3A")
ax.set_xlabel("waktu (detik)"); ax.set_ylabel("sudut lutut (°)")
ax.legend(frameon=False, loc="lower left")
ax.set_title(f"{n} repetisi", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Lihat hasilnya sebagai GIF: kerangka + sudut lutut + penghitung
gif = []
n_sejauh = 0
for i in range(0, len(SQUAT)):
    fr = SQUAT[i].copy()
    t = SEMUA_T[i]
    if t is not None:
        fr = gambar_kerangka(fr, t, SAMBUNG_POSE, tebal=2, r=3)
        gambar_sudut(fr, t, LUTUT_KANAN)
    n_sejauh += i in momen
    tulis(fr, f"Squat: {n_sejauh}", (15, 38), 1.0, EMAS)
    gif.append(fr)
tampil_gif(gif, "12_squat_counter.gif", fps=FPS_SQUAT, lebar=400, langkah=3)

**Perhatikan:**
- Video ini diambil dari **depan**. Sudut lutut dari depan **tidak sama** dengan sudut sebenarnya, karena lutut menekuk "ke arah kamera".
  Aplikasi fitness biasanya meminta kamera diletakkan di **samping**. (Ini proyeksi 3D ➜ 2D yang sama dengan Pertemuan 2.)
- Ambang terlalu sempit ➜ getaran kecil ikut dihitung. Ambang terlalu lebar ➜ squat yang dangkal tidak terhitung.

### 🔎 Sekilas: titik bahu sebagai sinyal (pemanasan untuk Tugas 3)
Setiap titik bisa dilihat sebagai **sinyal** terhadap waktu — persis seperti sinyal audio di awal semester.
Di bawah ini posisi tinggi **titik tengah bahu** (rata-rata titik 11 dan 12). Saat squat, bahu naik-turun puluhan piksel.

> Di **Tugas 3 (Respiratory Tracking)** Anda memakai titik yang sama untuk **napas**. Bedanya: gerak bahu saat bernapas hanya
> **beberapa piksel** dan bercampur dengan getaran deteksi. Bagaimana membuat sinyal sekecil itu terbaca — itu bagian
> yang harus Anda rancang sendiri. 😉

In [ ]:
bahu_y = np.array([np.nan if t is None else (t[11, 1] + t[12, 1]) / 2 for t in SEMUA_T])
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(waktu, bahu_y, color="#0B1F3A", lw=1.4)
ax.invert_yaxis()                        # y piksel bertambah ke BAWAH, jadi dibalik agar "naik" = naik
ax.set_xlabel("waktu (detik)"); ax.set_ylabel("tinggi bahu (piksel)")
ax.set_title("Titik tengah bahu (rata-rata titik 11 & 12)", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 4. Tangan Jadi 21 Titik (Hand Landmarker)

Satu tangan = **21 titik**: 1 titik pergelangan + **4 titik untuk tiap jari** (pangkal ➜ ujung).

| Jari | Titik (pangkal ➜ ujung) | Ujung |
|---|---|---|
| pergelangan | 0 | — |
| jempol | 1, 2, 3, 4 | **4** |
| telunjuk | 5, 6, 7, 8 | **8** |
| tengah | 9, 10, 11, 12 | **12** |
| manis | 13, 14, 15, 16 | **16** |
| kelingking | 17, 18, 19, 20 | **20** |

Pola mudah: **ujung jari = kelipatan 4** (4, 8, 12, 16, 20).

Seperti detektor wajah + landmark, Hand Landmarker bekerja dua tahap: **cari telapak tangan** dulu (kotak),
baru **taruh 21 titik** di dalamnya.

In [ ]:
SAMBUNG_TANGAN = [(0, 1), (1, 2), (2, 3), (3, 4),            # jempol
                  (0, 5), (5, 6), (6, 7), (7, 8),            # telunjuk
                  (5, 9), (9, 10), (10, 11), (11, 12),       # tengah
                  (9, 13), (13, 14), (14, 15), (15, 16),     # manis
                  (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)]   # kelingking + telapak
WARNA_JARI = [MERAH, ORANYE, EMAS, HIJAU, BIRU]              # jempol, telunjuk, tengah, manis, kelingking


def buat_tangan(mode="IMAGE", n=2):
    opsi = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=P("model/hand_landmarker.task")),
        running_mode=getattr(vision.RunningMode, mode), num_hands=n,
        min_hand_detection_confidence=0.5)
    return vision.HandLandmarker.create_from_options(opsi)


def titik_tangan(hasil, w, h):
    # -> list of (array (21, 2) piksel, label 'Left'/'Right', skor)
    out = []
    for lm, hd in zip(hasil.hand_landmarks, hasil.handedness):
        out.append((np.array([[p.x * w, p.y * h] for p in lm], np.float32), hd[0].category_name, hd[0].score))
    return out


def gambar_tangan(img, t, tebal=3, nomor=False):
    out = img.copy()
    for a, b in SAMBUNG_TANGAN:
        jari = 0 if b <= 4 else min((b - 1) // 4, 4)
        cv2.line(out, tuple(np.int32(t[a])), tuple(np.int32(t[b])), WARNA_JARI[jari], tebal, cv2.LINE_AA)
    for i, p in enumerate(t):
        cv2.circle(out, tuple(np.int32(p)), max(3, tebal + 1), (255, 255, 255), -1, cv2.LINE_AA)
        if nomor:
            tulis(out, str(i), (int(p[0]) + 5, int(p[1]) - 5), 0.4, EMAS)
    return out


tangan_img = buat_tangan("IMAGE")
ASTRO = cv2.imread(P("image/astronaut_melambai.jpg"))
s = 1000 / ASTRO.shape[1]
ASTRO = cv2.resize(ASTRO, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
ha, wa = ASTRO.shape[:2]
daftar = titik_tangan(tangan_img.detect(ke_mp(ASTRO)), wa, ha)
print(f"tangan ditemukan: {len(daftar)}")
hasil_astro = ASTRO.copy()
for t, label, skor in daftar:
    hasil_astro = gambar_tangan(hasil_astro, t, nomor=True)
    tulis(hasil_astro, f"{label} ({skor:.2f})", (int(t[:, 0].min()), int(t[:, 1].min()) - 12), 0.7, EMAS)
jajar([(ASTRO, "Chris Cassidy melambai (NASA)"), (hasil_astro, "21 titik tangan")], 6)

### Kiri atau kanan? Awas cermin!
MediaPipe menebak label `Left`/`Right` **dengan anggapan gambarnya dicerminkan** — seperti kamera depan HP/laptop yang
biasanya di-*flip*. Jadi untuk foto biasa (tidak dicerminkan), labelnya bisa **terbalik**.
Coba lihat: apa label tangan yang sama setelah gambarnya dicerminkan?

In [ ]:
cermin = cv2.flip(ASTRO, 1)
for nama, img in [("asli", ASTRO), ("dicerminkan", cermin)]:
    d = titik_tangan(tangan_img.detect(ke_mp(img)), wa, ha)
    print(f"{nama:12s} ->", [f"{lab} ({sk:.2f})" for _, lab, sk in d])
print("\nCassidy melambai dengan tangan KANANnya. Label mana yang benar?")

---
## 5. Menghitung Jari yang Terangkat

Sekarang bagian serunya: dari 21 titik, **berapa jari yang terangkat?**

### Aturan versi 1 (paling sederhana): bandingkan tinggi
Jari terangkat kalau **ujungnya lebih tinggi** dari sendi tengahnya, misalnya telunjuk: titik **8** di atas titik **6**.
Ingat: di gambar, "lebih tinggi" = nilai **y lebih kecil**.

### Aturan versi 2 (lebih tahan miring): bandingkan jarak ke pergelangan
Jari terangkat kalau **ujungnya lebih jauh dari pergelangan (titik 0)** daripada sendi tengahnya.
Jari yang menekuk melipat ujungnya kembali **mendekati** telapak.

**Jempol** beda sendiri karena ia menekuk ke **samping**, bukan ke bawah. Aturannya:
jempol terangkat kalau ujungnya (4) **lebih jauh dari pangkal kelingking (17)** daripada sendinya (3).

In [ ]:
UJUNG = [8, 12, 16, 20]       # telunjuk, tengah, manis, kelingking
SENDI = [6, 10, 14, 18]       # sendi tengah masing-masing jari


def jarak(t, i, j):
    return float(np.linalg.norm(t[i] - t[j]))


def jari_versi1(t):
    # bandingkan tinggi (y kecil = lebih tinggi). Jempol: pakai aturan jarak (sama di dua versi)
    jempol = jarak(t, 4, 17) > jarak(t, 3, 17)
    return [jempol] + [t[u, 1] < t[s, 1] for u, s in zip(UJUNG, SENDI)]


def jari_versi2(t):
    # bandingkan jarak ke pergelangan (titik 0) -> tidak peduli tangan tegak atau miring
    jempol = jarak(t, 4, 17) > jarak(t, 3, 17)
    return [jempol] + [jarak(t, u, 0) > jarak(t, s, 0) for u, s in zip(UJUNG, SENDI)]


NAMA_JARI = ["jempol", "telunjuk", "tengah", "manis", "kelingking"]


def potong_panel(img, n=5):
    w = img.shape[1] // n
    return [img[:, k * w:(k + 1) * w] for k in range(n)]


def kecilkan(img, tinggi=480):
    s = tinggi / img.shape[0]
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)


DERET = {
    "dari jempol": ("image/jari_dari_jempol.png", [1, 2, 3, 4, 5]),
    "dari telunjuk": ("image/jari_dari_telunjuk.png", [1, 2, 3, 4, 5]),
    "menekuk": ("image/jari_menekuk.png", [4, 3, 2, 1, 0]),
}
print("Jawaban benar = jumlah jari terangkat di tiap foto (lihat angka di pojok foto)")

### 🎬 Demo Dosen — uji aturan di 15 foto tangan
**Tebak dulu:** di deret mana aturan kita paling sering salah? (Petunjuk: perhatikan cara menekuk **jempol**.)

In [ ]:
rekap = []
for judul, (path, benar) in DERET.items():
    panel = [kecilkan(p) for p in potong_panel(cv2.imread(P(path)))]
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.6))
    for k, (pn, b) in enumerate(zip(panel, benar)):
        hh, ww = pn.shape[:2]
        d = titik_tangan(tangan_img.detect(ke_mp(pn)), ww, hh)
        if not d:
            tampil(axes[k], pn, "tangan tidak ketemu"); rekap.append((judul, b, None, None)); continue
        t = d[0][0]
        v1, v2 = sum(jari_versi1(t)), sum(jari_versi2(t))
        rekap.append((judul, b, v1, v2))
        warna = "#2E9E6B" if v2 == b else "#C8102E"
        tampil(axes[k], gambar_tangan(pn, t, tebal=2), f"benar {b} | program {v2}")
        axes[k].title.set_color(warna)
    fig.suptitle(f"Menghitung {judul}", fontweight="bold"); plt.tight_layout(); plt.show()

ok1 = sum(r[2] == r[1] for r in rekap); ok2 = sum(r[3] == r[1] for r in rekap)
print(f"Aturan versi 1 (tinggi): {ok1}/{len(rekap)} benar")
print(f"Aturan versi 2 (jarak) : {ok2}/{len(rekap)} benar")

### Uji ketahanan: bagaimana kalau tangannya miring?
Foto-foto tadi semuanya **tegak**. Sekarang tiap foto **diputar 90°** (tangan menunjuk ke samping).
Aturan mana yang tetap benar?

In [ ]:
# ✏️ Coba Ubah: arah putar (cv2.ROTATE_90_CLOCKWISE / cv2.ROTATE_180 / cv2.ROTATE_90_COUNTERCLOCKWISE)
PUTAR = cv2.ROTATE_90_CLOCKWISE

benar1 = benar2 = total = 0
contoh = []
for judul, (path, benar) in DERET.items():
    for pn, b in zip(potong_panel(cv2.imread(P(path))), benar):
        pn = cv2.rotate(kecilkan(pn), PUTAR)
        hh, ww = pn.shape[:2]
        d = titik_tangan(tangan_img.detect(ke_mp(pn)), ww, hh)
        if not d:
            continue
        t = d[0][0]
        v1, v2 = sum(jari_versi1(t)), sum(jari_versi2(t))
        benar1 += v1 == b; benar2 += v2 == b; total += 1
        if len(contoh) < 4 and judul == "dari jempol":
            contoh.append((gambar_tangan(pn, t, tebal=2), f"benar {b} | v1 {v1} | v2 {v2}"))
print(f"Setelah diputar: versi 1 benar {benar1}/{total}, versi 2 benar {benar2}/{total}")
jajar(contoh, 3.5)

**Perhatikan:**
- Versi 1 bergantung pada arah "atas". Begitu tangan miring, "lebih tinggi" tidak lagi berarti "terangkat".
- Versi 2 hanya memakai **jarak antar titik**, yang tidak berubah walau gambar diputar. Pelajaran yang sama dengan EAR minggu lalu:
  **pakai perbandingan jarak**, bukan posisi mentah.
- Jempol tetap yang paling sulit. Cara orang menekuk jempol sangat beragam (lihat deret "dari telunjuk").

---
## 6. Gesture Recognizer Bawaan MediaPipe

MediaPipe juga punya model yang **langsung menebak nama gestur** dari 21 titik — dilatih dari ribuan contoh, bukan aturan buatan tangan.
Ada 7 gestur bawaan (+ `None` kalau tidak yakin):

| Label | Arti | Label | Arti |
|---|---|---|---|
| `Closed_Fist` | ✊ kepal | `Thumb_Up` | 👍 jempol ke atas |
| `Open_Palm` | 🖐️ telapak terbuka | `Thumb_Down` | 👎 jempol ke bawah |
| `Pointing_Up` | ☝️ menunjuk ke atas | `Victory` | ✌️ huruf V |
| `ILoveYou` | 🤟 bahasa isyarat "I love you" | `None` | tidak dikenali |

In [ ]:
opsi_g = vision.GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=P("model/gesture_recognizer.task")),
    running_mode=vision.RunningMode.IMAGE, num_hands=1)
gestur = vision.GestureRecognizer.create_from_options(opsi_g)

for judul, (path, benar) in DERET.items():
    baris = []
    for pn, b in zip(potong_panel(cv2.imread(P(path))), benar):
        h_ = gestur.recognize(ke_mp(kecilkan(pn)))
        if h_.gestures:
            g = h_.gestures[0][0]
            baris.append(f"{b} jari -> {g.category_name} ({g.score:.2f})")
        else:
            baris.append(f"{b} jari -> (tangan tidak ketemu)")
    print(f"[{judul}]"); print("   " + "\n   ".join(baris))

g_astro = gestur.recognize(ke_mp(ASTRO))
if g_astro.gestures:
    print("\nAstronaut melambai ->", g_astro.gestures[0][0].category_name, f"({g_astro.gestures[0][0].score:.2f})")

**Perhatikan:**
- Model bawaan hanya kenal **7 gestur**. "Tiga jari" atau "empat jari" tidak ada di daftarnya, jadi hasilnya `None` atau tebakan terdekat.
- Aturan buatan sendiri (Bagian 5) bisa menghitung **0–5 jari**, tetapi tidak tahu "jempol ke atas" vs "jempol ke bawah".
- Di aplikasi nyata keduanya sering **digabung**: model untuk gestur umum, aturan sendiri untuk kebutuhan khusus.

---
## 7. Menghitung Jari di Video

Sekarang di video: seorang perempuan menghitung 1 sampai 10 dengan jarinya. Kita hitung jari terangkat **di setiap frame**
(semua tangan yang terlihat dijumlahkan).

### 🎬 Demo Dosen
**Tebak dulu:** apakah grafiknya akan naik rapi seperti tangga 1, 2, 3, …, 10? Atau ada "duri"?

In [ ]:
JARI_V, FPS_J = baca_video(P("video/hitung_jari.webm"), lebar=480)
hj, wj = JARI_V[0].shape[:2]
tangan_vid = buat_tangan("VIDEO", n=2)
JT = []                                       # list tangan tiap frame
for i, fr in enumerate(JARI_V):
    JT.append(titik_tangan(tangan_vid.detect_for_video(ke_mp(fr), int(i * 1000 / FPS_J)), wj, hj))
tangan_vid.close()

mentah = np.array([sum(sum(jari_versi2(t)) for t, _, _ in d) for d in JT])


def haluskan(deret, n=7):
    # ambil nilai yang PALING SERING muncul di n frame terakhir (modus) -> duri sesaat hilang
    jendela, out = deque(maxlen=n), []
    for v in deret:
        jendela.append(int(v))
        out.append(Counter(jendela).most_common(1)[0][0])
    return np.array(out)


# ✏️ Coba Ubah: panjang jendela penghalus (1 = tanpa penghalus)
N_HALUS = 7
halus = haluskan(mentah, N_HALUS)
tj = np.arange(len(mentah)) / FPS_J
fig, ax = plt.subplots(figsize=(12, 3.4))
ax.plot(tj, mentah, color="#D8D8D8", lw=1.2, label="mentah (per frame)")
ax.step(tj, halus, color="#C8102E", lw=2, where="post", label=f"dihaluskan (modus {N_HALUS} frame)")
ax.set_yticks(range(0, 11)); ax.set_xlabel("waktu (detik)"); ax.set_ylabel("jari terangkat")
ax.legend(frameon=False, loc="upper left"); plt.tight_layout(); plt.show()
print(f"{len(JARI_V)} frame @ {FPS_J:.2f} fps. Nilai berubah {np.sum(np.diff(mentah) != 0)} kali (mentah) "
      f"vs {np.sum(np.diff(halus) != 0)} kali (halus)")

In [ ]:
gif = []
for i, (fr, d) in enumerate(zip(JARI_V, JT)):
    fr = fr.copy()
    for t, _, _ in d:
        fr = gambar_tangan(fr, t, tebal=2)
    tulis(fr, f"Jari: {halus[i]}", (15, 45), 1.3, EMAS)
    gif.append(fr)
tampil_gif(gif, "12_hitung_jari.gif", fps=FPS_J, lebar=400, langkah=2)

**Perhatikan:**
- "Duri" muncul saat jari **sedang bergerak**: di tengah gerakan, jari setengah terangkat sehingga aturan ragu.
- Penghalus **modus** menunggu beberapa frame sampai jawabannya konsisten. Harganya: angka di layar **sedikit terlambat**.
  Sama seperti syarat "minimal 2 frame" pada kedip minggu lalu.

---
## 8. Virtual Painter: Telunjuk Jadi Kuas

Semua bahan sudah ada. Aturannya:

| Gestur | Aksi |
|---|---|
| ☝️ **hanya telunjuk** terangkat | **menggambar** — ujung telunjuk (titik 8) menjadi kuas |
| ✌️ **telunjuk + tengah** | **mode pilih** — kuas diangkat, arahkan ke kotak warna di atas layar |
| ✊ **kepal** selama 1 detik | **hapus** semua coretan |

Coretan disimpan di **kanvas terpisah** (gambar hitam seukuran frame), lalu ditempel ke frame kamera di setiap frame —
mirip *overlay* kacamata minggu lalu.

Di notebook kita coba versi kecil memakai video Bagian 7: setiap kali **hanya telunjuk** yang terangkat, ujungnya meninggalkan jejak.
Versi lengkap dengan webcam ada di **`12_virtual_painter.py`**.

In [ ]:
def hanya_telunjuk(jari):
    # jari = [jempol, telunjuk, tengah, manis, kelingking] (True = terangkat); jempol diabaikan
    return jari[1] and not (jari[2] or jari[3] or jari[4])


kanvas = np.zeros_like(JARI_V[0])
sebelum, gif = None, []
HALUS_KUAS = 0.5                  # ✏️ Coba Ubah: 0 = sangat halus (lambat), 1 = tanpa penghalus (bergetar)
for fr, d in zip(JARI_V, JT):
    fr = fr.copy()
    gambar = False
    if d:
        t = d[0][0]
        if hanya_telunjuk(jari_versi2(t)):
            ujung = t[8]
            if sebelum is not None:
                ujung = HALUS_KUAS * ujung + (1 - HALUS_KUAS) * sebelum      # rata-rata dengan posisi sebelumnya
                cv2.line(kanvas, tuple(np.int32(sebelum)), tuple(np.int32(ujung)), MERAH, 6, cv2.LINE_AA)
            sebelum, gambar = ujung, True
            cv2.circle(fr, tuple(np.int32(ujung)), 10, MERAH, -1, cv2.LINE_AA)
    if not gambar:
        sebelum = None                                                         # kuas diangkat
    topeng = kanvas.any(axis=2)
    fr[topeng] = kanvas[topeng]
    tulis(fr, "MENGGAMBAR" if gambar else "kuas diangkat", (15, 40), 0.9, MERAH if gambar else (255, 255, 255))
    gif.append(fr)

jajar([(gif[-1], "Frame terakhir + coretan"), (kanvas, "Isi kanvas")], 6)
tampil_gif(gif, "12_painter_mini.gif", fps=FPS_J, lebar=400, langkah=2)
print("Di video ini telunjuk sendirian hanya muncul sebentar (saat angka 1 dan 6), jadi coretannya pendek.")
print("Versi webcam: python 12_virtual_painter.py")

---
## 9. Dua Script Live (butuh webcam)

Jalankan dari folder `notebook_handson` di laptop Anda:

```bash
python 12_pose_sudut.py            # kerangka tubuh + sudut siku + penghitung angkat beban (bicep curl)
python 12_virtual_painter.py       # melukis di udara dengan telunjuk
```

| Script | Tombol |
|---|---|
| `12_pose_sudut.py` | `q` keluar · `m` ganti mode siku/lutut · `r` nolkan penghitung |
| `12_virtual_painter.py` | `q` keluar · `c` hapus kanvas · `s` simpan lukisan ke PNG · `t` titik tangan on/off |

Keduanya juga bisa membaca file video: `python 12_pose_sudut.py video/latihan_squat.webm`.
Model diunduh otomatis kalau belum ada di folder `model/`.

> ⚠️ Kalau webcam tidak terbuka di macOS, beri izin kamera untuk Terminal/VS Code di
> *System Settings ➜ Privacy & Security ➜ Camera*.

---
## 10. Rangkuman

1. **Pose Landmarker** memberi **33 titik tubuh** (ganjil = kiri, genap = kanan pemilik tubuh) + `visibility`.
2. **Sudut sendi** = sudut di titik tengah dari tiga titik (bahu–siku–pergelangan, pinggul–lutut–pergelangan kaki).
3. **Repetisi** dihitung dengan **dua ambang** (turun di bawah, naik di atas) agar getaran tidak ikut terhitung.
4. **Hand Landmarker** memberi **21 titik tangan**; ujung jari = 4, 8, 12, 16, 20. Label kiri/kanan menganggap gambar **dicerminkan**.
5. **Hitung jari**: ujung lebih jauh dari pergelangan daripada sendinya ➜ terangkat. Pakai **jarak**, bukan tinggi, agar tahan miring.
6. **Gesture Recognizer** mengenali 7 gestur bawaan; aturan sendiri cocok untuk kebutuhan khusus.
7. **Penghalus** (modus beberapa frame, rata-rata posisi) mengurangi getaran, dengan harga sedikit terlambat.
8. **Virtual Painter** = hitung jari + kanvas terpisah + overlay.

### 🧭 Lembar Contekan
| Tujuan | Kode |
|---|---|
| Pose | `vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=...)))` |
| Titik pose → piksel | `[[p.x * w, p.y * h, p.visibility] for p in hasil.pose_landmarks[0]]` |
| Tangan | `vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(..., num_hands=2))` |
| Label tangan | `hasil.handedness[i][0].category_name  # 'Left' / 'Right'` |
| Gestur bawaan | `vision.GestureRecognizer...recognize(img).gestures[0][0].category_name` |
| Mode video | `running_mode=vision.RunningMode.VIDEO` lalu `detect_for_video(img, waktu_ms)` |
| Sudut sendi | `sudut(titik[23], titik[25], titik[27])  # lutut kiri` |
| Jari terangkat | `jarak(ujung, pergelangan) > jarak(sendi, pergelangan)` |

> 🔜 **Pertemuan berikutnya:** rPPG — mengukur detak jantung dari warna wajah lewat webcam (metode POS).